In [ ]:
# math: Python 标准数学库，这里用 math.sqrt 做缩放点积中的开方
import math

# 导入 PyTorch 主包
import torch

# 导入 nn：包含 Module、Linear、Softmax 等神经网络组件
from torch import nn


In [ ]:
# torch.rand(128, 32, 512): 生成随机输入张量
# 维度通常解释为 [batch_size, seq_len, d_model]
x = torch.rand(128, 32, 512)


In [ ]:

# 自定义类 MultiHeadAttention：
# 作用：实现 Transformer 的多头注意力计算。
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        # 初始化父类
        super(MultiHeadAttention, self).__init__()
        self.n_head = n_head
        self.d_model = d_model

        # 错误写法（已注释）：
        # 问题：若 d_model 不能被 n_head 整除，后续 view 会报错且不易定位。
        # n_d = self.d_model // self.n_head

        # 正确做法：提前做显式检查，错误信息更清晰。
        if d_model % n_head != 0:
            raise ValueError(f'd_model({d_model}) 必须能被 n_head({n_head}) 整除。')

        # nn.Linear: 线性变换层，用于生成 Q/K/V
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # 多头拼接后的输出线性层
        self.w_concat = nn.Linear(d_model, d_model)

        # nn.Softmax(dim=-1): 在最后一维归一化为概率分布
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, q, k, v, mask=None):
        """
        自定义函数作用：执行多头注意力前向传播。

        参数：
        - q, k, v: 查询/键/值张量（常见形状 [batch, seq_len, d_model]）
        - mask: 可选掩码，0 表示被屏蔽位置

        返回：
        - out: 注意力输出（形状 [batch, q_len, d_model]）
        """
        # 错误写法（已注释）：
        # 问题：只从 q 读取 time，然后用于 k/v 的 view。
        # 在交叉注意力（q_len != k_len）时会形状错误。
        # batch, time, dimension = q.shape

        # 正确写法：分别读取 q/k/v 的序列长度，更通用。
        batch, q_len, dimension = q.shape
        _, k_len, _ = k.shape
        _, v_len, _ = v.shape

        # 保证键和值长度一致（标准 attention 要求）
        if k_len != v_len:
            raise ValueError(f'k_len({k_len}) 与 v_len({v_len}) 必须一致。')

        # 每个头的维度 = d_model / n_head
        n_d = self.d_model // self.n_head

        # 先线性投影得到 Q/K/V
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)

        # view: 拆分头维度；permute: 调整为 [batch, head, time, head_dim]
        q = q.view(batch, q_len, self.n_head, n_d).permute(0, 2, 1, 3)
        k = k.view(batch, k_len, self.n_head, n_d).permute(0, 2, 1, 3)
        v = v.view(batch, v_len, self.n_head, n_d).permute(0, 2, 1, 3)

        # 缩放点积注意力：QK^T / sqrt(head_dim)
        score = q @ k.transpose(2, 3) / math.sqrt(n_d)

        # masked_fill: 在 mask==0 的位置填极小值，softmax 后接近 0
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e9)

        # softmax 得到权重，再与 V 相乘得到上下文
        score = self.softmax(score) @ v

        # permute + contiguous + view: 恢复为 [batch, q_len, d_model]
        score = score.permute(0, 2, 1, 3).contiguous().view(batch, q_len, dimension)

        # 最终线性映射融合多头信息
        out = self.w_concat(score)
        return out


In [ ]:
# ========================
# 单元测试（MultiHeadAttention 相关）
# ========================

print("\n[测试开始] attention.ipynb 逻辑单元测试")
print("说明：本单元会测试 MultiHeadAttention 的参数检查、前向形状、交叉注意力与 mask 行为。")

test_results = []


def run_test(test_name, fn):
    print(f"\n[测试项] {test_name}")
    try:
        fn()
        print(f"[结果] {test_name}: 通过")
        test_results.append((test_name, True, ""))
    except Exception as e:
        print(f"[结果] {test_name}: 失败")
        print(f"[错误信息] {type(e).__name__}: {e}")
        test_results.append((test_name, False, f"{type(e).__name__}: {e}"))


def test_divisibility_check():
    print("  - 步骤1：构造非法参数 d_model=10, n_head=3")
    print("  - 步骤2：检查是否抛出 ValueError")
    try:
        _ = MultiHeadAttention(d_model=10, n_head=3)
    except ValueError:
        return
    raise AssertionError("期望抛出 ValueError，但没有抛出")



def test_self_attention_output_shape():
    print("  - 步骤1：实例化 MultiHeadAttention(d_model=16, n_head=4)")
    mha = MultiHeadAttention(d_model=16, n_head=4)

    print("  - 步骤2：构造自注意力输入 q=k=v，形状 [2, 6, 16]")
    q = torch.randn(2, 6, 16)

    print("  - 步骤3：前向计算并检查输出形状")
    out = mha(q, q, q)
    assert out.shape == (2, 6, 16), f"期望 (2, 6, 16)，实际 {tuple(out.shape)}"



def test_cross_attention_shape_support():
    print("  - 步骤1：构造交叉注意力输入 q_len=3, k_len=v_len=5")
    mha = MultiHeadAttention(d_model=16, n_head=4)
    q = torch.randn(2, 3, 16)
    k = torch.randn(2, 5, 16)
    v = torch.randn(2, 5, 16)

    print("  - 步骤2：前向计算并检查输出形状是否为 [2, 3, 16]")
    out = mha(q, k, v)
    assert out.shape == (2, 3, 16), f"期望 (2, 3, 16)，实际 {tuple(out.shape)}"



def test_kv_length_mismatch_check():
    print("  - 步骤1：构造 k_len != v_len 的非法输入")
    mha = MultiHeadAttention(d_model=16, n_head=4)
    q = torch.randn(2, 3, 16)
    k = torch.randn(2, 5, 16)
    v = torch.randn(2, 4, 16)

    print("  - 步骤2：检查 forward 是否抛出 ValueError")
    try:
        _ = mha(q, k, v)
    except ValueError:
        return
    raise AssertionError("期望抛出 ValueError，但没有抛出")



def test_mask_effect():
    print("  - 步骤1：构造 q/k/v 和 mask（屏蔽最后一个 key 位置）")
    mha = MultiHeadAttention(d_model=16, n_head=4)
    q = torch.randn(1, 3, 16)
    k = torch.randn(1, 4, 16)
    v = torch.randn(1, 4, 16)

    # mask 形状可广播到 [batch, head, q_len, k_len]
    mask = torch.ones(1, 1, 1, 4)
    mask[..., -1] = 0

    print("  - 步骤2：分别计算无 mask 与有 mask 的输出")
    out_no_mask = mha(q, k, v, mask=None)
    out_with_mask = mha(q, k, v, mask=mask)

    print("  - 步骤3：检查 mask 确实影响输出")
    assert not torch.allclose(out_no_mask, out_with_mask), "mask 生效后输出应发生变化"


run_test("参数整除检查测试（d_model % n_head）", test_divisibility_check)
run_test("自注意力输出形状测试", test_self_attention_output_shape)
run_test("交叉注意力形状支持测试", test_cross_attention_shape_support)
run_test("k/v 长度不一致报错测试", test_kv_length_mismatch_check)
run_test("mask 生效测试", test_mask_effect)

print("\n[测试汇总]")
passed = sum(1 for _, ok, _ in test_results if ok)
failed = len(test_results) - passed
for name, ok, err in test_results:
    status = "通过" if ok else "失败"
    print(f"- {name}: {status}")
    if err:
        print(f"  错误详情: {err}")

print(f"总计: {len(test_results)} 项, 通过: {passed}, 失败: {failed}")

if failed == 0:
    print("[最终结果] 所有测试通过，当前 notebook 上述逻辑正确。")
else:
    raise AssertionError("存在测试失败，请根据上方错误详情检查对应逻辑。")
